# Entrenamiento de detección de emociones con Eigenfaces + SVM

Importación de paquetes

In [1]:
import os

import cv2
import joblib
import numpy as np
from joblib import Memory
from sklearn.decomposition import PCA
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

Variables del dataset

In [2]:
DATASET_DIR = "dataset_processed"
IMG_SIZE = 32
MODEL_PATH = "model_results/eigenfaces_svm.joblib"

CLASSES = sorted(os.listdir(os.path.join(DATASET_DIR, "train")))
print("Clases:", CLASSES)

Clases: ['Angry', 'Fear', 'Happy', 'Sad', 'Suprise']


Cargar imágenes en gris y 64x64

In [3]:
def load_split(split_name):
    X, y = [], []
    split_dir = os.path.join(DATASET_DIR, split_name)
    for label in CLASSES:
        class_dir = os.path.join(split_dir, label)
        for fname in sorted(os.listdir(class_dir)):
            img = cv2.imread(os.path.join(class_dir, fname), cv2.IMREAD_GRAYSCALE)
            img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
            X.append(img.flatten())
            y.append(label)
    return np.array(X), np.array(y)


X_train, y_train = load_split("train")
X_test, y_test = load_split("test")

print("Train:", X_train.shape, "Test:", X_test.shape)

Train: (39100, 1024) Test: (9825, 1024)


Entrenamiento

In [4]:
cache_dir = "pipeline_cache"
memory = Memory(location=cache_dir, verbose=0)

pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("pca", PCA(random_state=42)),
    ("svm", SVC()),
], memory=memory)

param_grid = {
    "pca__n_components": [50, 100, 150],
    "svm__C": [1, 10, 100],
    "svm__kernel": ["rbf"],
}

grid_search = GridSearchCV(pipeline, param_grid, cv=3, n_jobs=2, verbose=1)
grid_search.fit(X_train, y_train)

print("Mejores parámetros:", grid_search.best_params_)
print("Mejor accuracy en validación cruzada:", grid_search.best_score_)

Fitting 3 folds for each of 9 candidates, totalling 27 fits
Mejores parámetros: {'pca__n_components': 150, 'svm__C': 100, 'svm__kernel': 'rbf'}
Mejor accuracy en validación cruzada: 0.8035290785775118


Guardar el modelo entrenado

In [6]:
best_model = grid_search.best_estimator_
joblib.dump(best_model, MODEL_PATH)
print(f"Modelo guardado en {MODEL_PATH}")

Modelo guardado en model_results/eigenfaces_svm.joblib
